# GeoLife CP3 — API Contract + Full-release HTTP Parity

**Mục tiêu:** validate rằng FastAPI layer của CP3 giữ nguyên frozen CP2 semantics khi đi qua HTTP/Pydantic/OpenAPI, đồng thời không làm rò precise inferred Home/Office coordinates trong response.

Notebook này không train/tune model mới. Nó trả lời các câu hỏi serving:

1. health/OpenAPI contract có đúng như design không;
2. request stay events có được normalize thành đúng production model input không;
3. full-release HTTP inference có reproduce đúng `27 HOME / 16 OFFICE` không;
4. emitted evidence có parity với `infer_home_office()` trực tiếp không;
5. abstention có được biểu diễn như model output `200`, thay vì HTTP error không;
6. validation response có tránh echo location history nhạy cảm không.

> Input v1 là **CP1 stay events của một user/request**, không phải raw GPS. API không expose threshold overrides và không trả precise inferred semantic coordinates.


## Cách chạy

Notebook reuse private cache đã materialize ở CP2:

```text
/mnt/geolife-data/cache/cp2_home_office/stays_baseline_v1.pkl
```

Vì vậy **không rerun 18,670 raw trajectory files**. Nó chỉ recompute semantic inference và gọi FastAPI in-process qua `TestClient`.

Trong PR CP3, default repo branch là `cp3-api-serving`. Sau khi merge có thể set `GEOLIFE_REPO_BRANCH=main`.


In [ ]:
from pathlib import Path
from collections import Counter
from IPython.display import display
import os
import subprocess
import sys

import numpy as np
import pandas as pd
from fastapi.testclient import TestClient

REPO_URL = "https://github.com/tanh1c/geolife.git"
REPO_BRANCH = os.environ.get("GEOLIFE_REPO_BRANCH", "cp3-api-serving")
REPO_DIR = Path(os.environ.get("GEOLIFE_REPO_DIR", "/tmp/geolife"))
STAYS_CACHE = Path(os.environ.get(
    "GEOLIFE_CP2_STAYS_CACHE",
    "/mnt/geolife-data/cache/cp2_home_office/stays_baseline_v1.pkl",
))

def ensure_repo():
    if (REPO_DIR / ".git").exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH], check=True)
    else:
        subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)

ensure_repo()
src_dir = REPO_DIR / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from geolife.api.app import app
from geolife.model import infer_home_office

client = TestClient(app)

print("Repo:", REPO_DIR)
print("Branch:", REPO_BRANCH)
print("Stay cache:", STAYS_CACHE)


## 1. Reuse frozen CP2 full-release stay table

### Contract gate

CP3 phải consume đúng upstream table mà CP2 production parity đã dùng:

```text
5,821 stays
136 users with >=1 stay
```

Nếu cache không reconcile, dừng notebook. HTTP parity trên upstream khác không có ý nghĩa.


In [ ]:
if not STAYS_CACHE.exists():
    raise FileNotFoundError(
        f"Missing {STAYS_CACHE}. Run notebook 03 materialization first."
    )

stays = pd.read_pickle(STAYS_CACHE).copy()
stays["arrival_time_utc"] = pd.to_datetime(stays["arrival_time_utc"], utc=True)
stays["departure_time_utc"] = pd.to_datetime(stays["departure_time_utc"], utc=True)
stays["user_id"] = stays["user_id"].astype(str)

print("Stays:", len(stays))
print("Users:", stays["user_id"].nunique())

assert len(stays) == 5821
assert stays["user_id"].nunique() == 136


## 2. HTTP/OpenAPI smoke contract

Trước full parity, kiểm tra serving surface nhỏ nhất:

- `/health` trả service + API/model contract version;
- OpenAPI có `/v1/home-office/infer`;
- v1 chỉ expose frozen model semantics, không expose tuning params.


In [ ]:
health = client.get("/health")
assert health.status_code == 200
display(health.json())

schema = client.get("/openapi.json").json()
print("API title:", schema["info"]["title"])
print("API version:", schema["info"]["version"])
print("Paths:")
for path in sorted(schema["paths"]):
    print(" -", path)

assert "/health" in schema["paths"]
assert "/v1/home-office/infer" in schema["paths"]

emitted_props = schema["components"]["schemas"]["EmittedResult"]["properties"]
assert "latitude" not in emitted_props
assert "longitude" not in emitted_props


### Cách đọc phần 2

OpenAPI là public contract của serving layer. Nếu precise semantic coordinates xuất hiện trong `EmittedResult`, đó là contract regression ngay cả khi Python route code vẫn “đúng”.

Tương tự, client không được truyền `home_min_share`, `location_max_diameter_m`, v.v. ở request v1; threshold override sẽ biến một frozen model contract thành per-request tuning surface.


## 3. Direct production baseline — source of parity truth

Trước khi đi qua HTTP, chạy production `infer_home_office(stays)` trực tiếp trên full cache.

Expected frozen CP2 parity:

```text
HOME    27
OFFICE  16
43 rows / 36 unique users
```

Đây là reference cho HTTP layer, không phải ground-truth accuracy.


In [ ]:
direct = infer_home_office(stays)
direct_counts = direct["label"].value_counts().reindex(["HOME", "OFFICE"], fill_value=0)

display(direct_counts.rename("direct_emitted_users").to_frame())
print("Rows:", len(direct))
print("Unique users:", direct["user_id"].nunique())

assert int(direct_counts["HOME"]) == 27
assert int(direct_counts["OFFICE"]) == 16
assert len(direct) == 43
assert direct["user_id"].nunique() == 36


## 4. Full-release HTTP parity — one user per request

API contract cố ý nhận **một user/request**. Ta vì vậy replay 136 users qua HTTP layer in-process.

Mỗi request bỏ `duration_s`; API phải derive duration từ arrival/departure để tránh hai nguồn thời gian mâu thuẫn.

Parity yêu cầu:

1. exact emitted `(user_id, label)` set giống direct model;
2. `location_id` giống;
3. share / margin / dates / dwell / evidence strength giống trong numerical tolerance;
4. aggregate HTTP counts vẫn là `27 / 16`.


In [ ]:
def iso_z(ts):
    return pd.Timestamp(ts).tz_convert("UTC").isoformat().replace("+00:00", "Z")

def user_payload(user_id, group):
    return {
        "user_id": str(user_id),
        "stays": [
            {
                "arrival_time_utc": iso_z(row.arrival_time_utc),
                "departure_time_utc": iso_z(row.departure_time_utc),
                "latitude": float(row.latitude),
                "longitude": float(row.longitude),
            }
            for row in group.itertuples(index=False)
        ],
    }

http_rows = []
abstentions = Counter()

for i, (user_id, group) in enumerate(stays.groupby("user_id", sort=True), start=1):
    response = client.post(
        "/v1/home-office/infer",
        json=user_payload(user_id, group),
    )
    assert response.status_code == 200, (user_id, response.text)

    body = response.json()
    assert body["user_id"] == str(user_id)
    assert body["model_contract"] == "cp2-v1"

    for result in body["results"]:
        if result["status"] == "emitted":
            http_rows.append({"user_id": str(user_id), **result})
        else:
            abstentions[(result["label"], result["reason"])] += 1

    if i % 25 == 0:
        print(f"{i}/136 users")

http_emitted = pd.DataFrame(http_rows)
http_counts = http_emitted["label"].value_counts().reindex(["HOME", "OFFICE"], fill_value=0)

display(http_counts.rename("http_emitted_users").to_frame())
print("HTTP emitted rows:", len(http_emitted))
print("HTTP unique users:", http_emitted["user_id"].nunique())

abstention_summary = pd.DataFrame(
    [
        {"label": label, "reason": reason, "count": count}
        for (label, reason), count in sorted(abstentions.items())
    ]
)
display(abstention_summary)

assert int(http_counts["HOME"]) == 27
assert int(http_counts["OFFICE"]) == 16
assert len(http_emitted) == 43
assert http_emitted["user_id"].nunique() == 36


In [ ]:
direct_cmp = direct[
    [
        "user_id", "label", "location_id", "evidence_strength",
        "relevant_dwell_share", "share_margin", "relevant_dates", "relevant_dwell_h",
    ]
].copy()
direct_cmp["user_id"] = direct_cmp["user_id"].astype(str)

http_cmp = http_emitted[
    [
        "user_id", "label", "location_id", "evidence_strength",
        "relevant_dwell_share", "share_margin", "relevant_dates", "relevant_dwell_h",
    ]
].copy()

direct_cmp = direct_cmp.sort_values(["user_id", "label"]).reset_index(drop=True)
http_cmp = http_cmp.sort_values(["user_id", "label"]).reset_index(drop=True)

assert direct_cmp[["user_id", "label"]].equals(http_cmp[["user_id", "label"]])
assert direct_cmp["location_id"].astype(int).equals(http_cmp["location_id"].astype(int))
assert direct_cmp["relevant_dates"].astype(int).equals(http_cmp["relevant_dates"].astype(int))

for col in [
    "evidence_strength",
    "relevant_dwell_share",
    "share_margin",
    "relevant_dwell_h",
]:
    assert np.allclose(
        direct_cmp[col].to_numpy(dtype=float),
        http_cmp[col].to_numpy(dtype=float),
        rtol=1e-12,
        atol=1e-12,
    ), col

print("Full-release HTTP ↔ direct-model parity: OK")


### Cách đọc full-release parity

Final measured run trên **5,821 stays / 136 users**:

| path | HOME | OFFICE | emitted rows | unique emitted users |
|---|---:|---:|---:|---:|
| direct production | 27 | 16 | 43 | 36 |
| HTTP replay | 27 | 16 | 43 | 36 |

Không chỉ aggregate counts match. Notebook còn PASS:

- exact emitted `(user_id, label)` set;
- `location_id`;
- `relevant_dates`;
- `evidence_strength`;
- `relevant_dwell_share`;
- `share_margin`;
- `relevant_dwell_h`.

Nếu aggregate `27/16` giống nhưng `(user,label)` set khác, API vẫn **fail parity** — tổng count có thể tình cờ cân bằng.

### Abstention observability

Measured valid-request outcomes:

| label | reason | count |
|---|---|---:|
| HOME | out_of_scope_geography | 39 |
| HOME | insufficient_recurring_history | 24 |
| HOME | insufficient_semantic_evidence | 46 |
| OFFICE | out_of_scope_geography | 39 |
| OFFICE | insufficient_recurring_history | 24 |
| OFFICE | insufficient_semantic_evidence | 57 |

Cộng emitted + abstained cho mỗi label đều bằng **136 requests**.

Các count này là **serving/model outcomes**, không phải error rate và không phải accuracy table.

### Warning discovered by full replay

Run đầu phát hiện pandas `FutureWarning` khi một user chỉ emit HOME hoặc chỉ OFFICE: production model concat một non-empty emission frame với một empty frame.

Prediction values vẫn đúng và parity vẫn PASS, nhưng warning báo future dtype-behavior risk. Production code sau audit đã đổi sang chỉ concat non-empty frames và thêm regression test.

Vì vậy full replay đã kiểm tra cả **semantic parity** lẫn một integration edge case mà deterministic unit fixture trước đó chưa làm lộ.

## 5. Validation + privacy contract smoke checks

Hai loại failure phải tách nhau:

```text
malformed request          → HTTP 422
valid request, weak model  → HTTP 200 + abstained
```

Validation error cũng không nên echo full GPS/stay values trong response body.


In [ ]:
bad_time = {
    "user_id": "privacy-probe",
    "stays": [{
        "arrival_time_utc": "2026-01-05T13:00:00",
        "departure_time_utc": "2026-01-05T14:00:00Z",
        "latitude": 39.912345,
        "longitude": 116.456789,
    }],
}
response = client.post("/v1/home-office/infer", json=bad_time)
assert response.status_code == 422
assert "39.912345" not in response.text
assert "116.456789" not in response.text

override = {
    "user_id": "override-probe",
    "stays": [{
        "arrival_time_utc": "2026-01-05T13:00:00Z",
        "departure_time_utc": "2026-01-05T14:00:00Z",
        "latitude": 39.90,
        "longitude": 116.40,
    }],
    "home_min_share": 0.0,
}
response = client.post("/v1/home-office/infer", json=override)
assert response.status_code == 422

print("Validation/privacy smoke checks: OK")


## 6. CP3 kết luận

Final full-release run đã đóng toàn bộ serving gates:

1. unit/API tests GREEN;
2. OpenAPI contract đúng design;
3. validation privacy checks PASS;
4. direct production = **27 HOME / 16 OFFICE**;
5. HTTP replay = **27 HOME / 16 OFFICE**;
6. cả hai đều = **43 emitted rows / 36 unique users**;
7. exact emitted user/label set parity;
8. location-id + evidence-field parity;
9. abstention explicit HTTP-200 model output;
10. no precise inferred coordinates in response schema.

### Frozen CP3 v1 serving contract

```text
CP1 stay events
      ↓
one user / HTTP request
      ↓
Pydantic validation
      ↓
frozen CP2 infer_home_office()
      ↓
HOME / OFFICE emitted-or-abstained response
```

### Điều CP3 không làm

- không retune CP2 model;
- không serve raw GPS cleaning synchronously;
- không gọi mọi OTHER location là POI;
- không biến evidence strength thành probability;
- không persist request location history;
- không expose precise inferred Home/Office coordinates.

### Status

**Full-release HTTP ↔ direct-model parity: PASS.**

Sau warning-cleanup + final CI, PR CP3 đủ điều kiện chuyển sang Ready for Review. Checkpoint tiếp theo có thể tập trung vào **deployment/container/observability** thay vì thay đổi model semantics.